# S4/S5 Augmentation Diagnostics

Diagnostic notebook for evaluating back-translation (S4) and T5 paraphrase (S5) augmentation quality on rare-bin training examples (bins 3–5 and 6+).

**Purpose:** Before committing to a full training run, check:
1. Do augmented narratives preserve the original fatality count? (count-preservation pass rate per language / strategy)
2. What do failures look like? (manual inspection)
3. Is S4 or S5 coverage sufficient to justify integration into the training pipeline?

See `papers/death-counts/notes/s4-s5-augmentation-strategy.md` and `rare-bin-attack-plan.md` for context.

## 1. Setup

### 1.1 Colab Setup

Mount Google Drive, clone the repo, install dependencies, and add required paths to `sys.path`.

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install dependencies
# transformers is pinned to 4.57.1 in count-models/requirements.txt.
# Colab pre-installs a newer version, so a runtime restart is required after install
# for the correct version to be active. This cell handles that automatically.
%pip install -qU pip setuptools wheel
%pip install -q -r /content/code-satp/models/count-models/requirements.txt

# 4) Paths
import pathlib, sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results/augmentation-diagnostics").mkdir(parents=True, exist_ok=True)
sys.path.append("/content/code-satp/models/classification-models/imbalance-handling")
sys.path.append("/content/code-satp/models/count-models/utils")

# 5) GPU check
import torch
print('=' * 60)
print('SETUP COMPLETE')
print('=' * 60)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected — T5 paraphrase generation will be slow.')

# 6) Verify transformers version and restart runtime if needed
import transformers
print(f'transformers version: {transformers.__version__} (expected: 4.57.1)')
if transformers.__version__ != '4.57.1':
    print('Restarting runtime to activate transformers==4.57.1...')
    print('After restart, skip this cell and run from the imports cell down.')
    import os
    os.kill(os.getpid(), 9)

### 1.2 Imports and Config

In [ ]:
import sys
import re
import time
import random
from pathlib import Path

import pandas as pd
import numpy as np

# ── Config ───────────────────────────────────────────────────────────────────
# Paths
MAIN_DATA_PATH = Path("/content/code-satp/data/satp_clean.csv")
VAL_PATH       = Path("/content/code-satp/models/count-models/data/val.csv")
TEST_PATH      = Path("/content/code-satp/models/count-models/data/test.csv")
RESULTS_DIR    = Path("/content/drive/MyDrive/colab/satp-results/augmentation-diagnostics")

# Sample size cap for the diagnostic.
# Set to None to run on all rare-bin examples (recommended for final diagnostics).
# Start with 50 for a quick first pass.
SAMPLE_N = None
RANDOM_SEED = 42

# T5 paraphrase model — flan-t5-large matches the model used in the classification paper
T5_PARAPHRASE_MODEL = "google/flan-t5-large"

# Diagnostic display / generation settings
INSPECT_N      = 15   # number of failure examples to display in inspection cells
N_PARAPHRASES  = 2    # T5 paraphrases generated per source example

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Config set.")

## 2. Load data and reconstruct train split

In [ ]:
def assign_bin(n):
    if n == 0:       return '0'
    elif n == 1:     return '1'
    elif n == 2:     return '2'
    elif n <= 5:     return '3-5'
    else:            return '6+'

# Load main dataset (same filter as the seq2seq notebook)
df_all = pd.read_csv(MAIN_DATA_PATH)
df_all = df_all[df_all['first_action'].isin(['Armed Assault', 'Bombing'])].copy()
df_all = df_all.dropna(subset=['incident_summary', 'total_fatalities'])
df_all['total_fatalities'] = df_all['total_fatalities'].astype(int)

# Load val and test splits
val_df  = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

held_out_ids = set(val_df['incident_number']) | set(test_df['incident_number'])
train_df = df_all[~df_all['incident_number'].isin(held_out_ids)].copy()
train_df['bin'] = train_df['total_fatalities'].apply(assign_bin)

print(f"Train set size: {len(train_df)}")
print("\nBin distribution in train set:")
print(train_df['bin'].value_counts().sort_index())

In [ ]:
# Filter to rare-bin examples only (bins 3-5 and 6+)
rare_df = train_df[train_df['bin'].isin(['3-5', '6+'])].copy().reset_index(drop=True)
print(f"Rare-bin examples (bins 3-5 and 6+): {len(rare_df)}")
print(rare_df['bin'].value_counts())

# Draw diagnostic sample
if SAMPLE_N is not None and SAMPLE_N < len(rare_df):
    sample_df = rare_df.sample(n=SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"\nUsing sample of {SAMPLE_N} examples for diagnostics")
else:
    sample_df = rare_df.copy()
    print(f"\nUsing all {len(rare_df)} rare-bin examples")

print(sample_df['bin'].value_counts())

## 3. Count-preservation check

For a raw narrative (not model output), we check whether the original fatality count numeral still appears in the augmented text as a word-boundary match. This is a text-level proxy — if the numeral has been dropped, altered, or converted to a word, the check fails.

Note: `parse_prediction()` from `extraction_utils.py` is designed for model output strings, not raw narratives. Using it here would return the first number found in the text, which could be an injury count rather than the fatality count. The regex approach below is more appropriate for source-text checking.

In [ ]:
def count_preserved(original_label: int, augmented_text: str) -> bool:
    """
    Check whether the original fatality count appears in the augmented text,
    either as a numeral or as an English word form.

    Uses word-boundary regex so '3' does not match '13' or '30'.
    Word forms covered up to 20 — counts above 20 appear as numerals in SATP narratives.
    For label == 0: always returns True (not used for augmentation but included for completeness).
    """
    if augmented_text is None or not isinstance(augmented_text, str) or len(augmented_text.strip()) == 0:
        return False
    if original_label == 0:
        return True
    # Check numeral form
    if re.search(r'\b' + re.escape(str(original_label)) + r'\b', augmented_text):
        return True
    # Check word form for small numbers (SATP narratives frequently spell these out)
    num_words = {
        1:'one', 2:'two', 3:'three', 4:'four', 5:'five',
        6:'six', 7:'seven', 8:'eight', 9:'nine', 10:'ten',
        11:'eleven', 12:'twelve', 13:'thirteen', 14:'fourteen', 15:'fifteen',
        16:'sixteen', 17:'seventeen', 18:'eighteen', 19:'nineteen', 20:'twenty'
    }
    word = num_words.get(original_label, '')
    if word and re.search(r'\b' + word + r'\b', augmented_text, re.IGNORECASE):
        return True
    return False


def run_preservation_check(results_df: pd.DataFrame) -> pd.DataFrame:
    """Add a 'preserved' column to a results dataframe.
    Expects columns: 'total_fatalities', 'augmented_text'."""
    results_df = results_df.copy()
    results_df['preserved'] = results_df.apply(
        lambda row: count_preserved(row['total_fatalities'], row['augmented_text']),
        axis=1
    )
    return results_df


def summarise_pass_rates(df, groupby_cols):
    """Aggregate preserved column by groupby_cols, returning n_total, n_pass, pass_rate."""
    return (
        df.groupby(groupby_cols)['preserved']
        .agg(n_total='count', n_pass='sum')
        .assign(pass_rate=lambda x: (x['n_pass'] / x['n_total'] * 100).round(1))
        .reset_index()
    )


# Quick sanity check
assert count_preserved(5, "five Maoists and 5 others were killed") == True
assert count_preserved(5, "Five Maoists were killed") == True       # word form now passes
assert count_preserved(3, "13 people were killed") == False          # word boundary prevents false match
assert count_preserved(3, "3 people were killed") == True
assert count_preserved(3, "Three people were killed") == True        # word form passes
print("Sanity checks passed.")

## 4. S4 — Back-Translation

Three pivot languages: Hindi (`hi`), Urdu (`ur`), Bengali (`bn`). Each example is translated to the pivot language and back to English. We instantiate a separate augmenter per language to get per-language pass rates.

Requires internet access. Adjust `API_DELAY_SECONDS` if rate limits are hit.

### 4.1 Initialise augmenters

In [ ]:
from imbalance_handling_strategies import BackTranslationAugmentation

# Instantiate one augmenter per pivot language for explicit per-language control
augmenters_bt = {
    'hi': BackTranslationAugmentation(target_languages=['hi']),
    'ur': BackTranslationAugmentation(target_languages=['ur']),
    'bn': BackTranslationAugmentation(target_languages=['bn']),
}

for lang, aug in augmenters_bt.items():
    status = "available" if aug.available else "NOT AVAILABLE"
    print(f"  {lang}: {status}")

### 4.2 Run back-translation diagnostic

In [ ]:
# ── Run back-translation diagnostic ─────────────────────────────────────────
API_DELAY_SECONDS = 0.5

bt_records = []

for lang, aug in augmenters_bt.items():
    if not aug.available:
        print(f"Skipping {lang} — augmenter not available")
        continue
    print(f"Running back-translation via {lang} ({len(sample_df)} examples)...")
    for _, row in sample_df.iterrows():
        try:
            augmented_texts = aug.augment_text(row['incident_summary'], num_augmentations=1)
            aug_text = augmented_texts[0] if augmented_texts else None
        except Exception as e:
            aug_text = None
        bt_records.append({
            'incident_number':  row['incident_number'],
            'bin':              row['bin'],
            'total_fatalities': row['total_fatalities'],
            'original_text':    row['incident_summary'],
            'pivot_language':   lang,
            'augmented_text':   aug_text,
        })
        time.sleep(API_DELAY_SECONDS)
    print(f"  Done.")

bt_df = pd.DataFrame(bt_records)
bt_df = run_preservation_check(bt_df)
print(f"\nTotal back-translation results: {len(bt_df)}")

### 4.3 Pass rates

In [ ]:
# ── S4 pass rates ────────────────────────────────────────────────────────────
print("=== S4 Pass rate by pivot language ===")
print(summarise_pass_rates(bt_df, ['pivot_language']).to_string(index=False))

print("\n=== S4 Pass rate by pivot language × bin ===")
print(summarise_pass_rates(bt_df, ['pivot_language', 'bin']).to_string(index=False))

### 4.4 Failure inspection

In [ ]:
# ── S4 failure inspection ────────────────────────────────────────────────────
bt_failures = bt_df[~bt_df['preserved']].sort_values(['pivot_language', 'total_fatalities'])
print(f"Total failures: {len(bt_failures)} of {len(bt_df)} ({100*len(bt_failures)/len(bt_df):.1f}%)")
print()

for _, row in bt_failures.head(INSPECT_N).iterrows():
    print(f"[{row['pivot_language']} | bin={row['bin']} | label={row['total_fatalities']}]")
    print(f"  ORIGINAL:  {row['original_text'][:200]}")
    aug = row['augmented_text']
    print(f"  AUGMENTED: {str(aug)[:200] if aug else 'TRANSLATION FAILED'}")
    print()

### 4.5 Passing example inspection

In [ ]:
# ── S4 passing example inspection (stratified sample) ───────────────────────
bt_passes = bt_df[bt_df['preserved']]
print(f"Passing examples: {len(bt_passes)} of {len(bt_df)}")
print()

inspect_passes = (
    bt_passes
    .groupby(['pivot_language', 'bin'], group_keys=False)
    .apply(lambda g: g.sample(min(2, len(g)), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

for _, row in inspect_passes.iterrows():
    print(f"[{row['pivot_language']} | bin={row['bin']} | label={row['total_fatalities']}]")
    print(f"  ORIGINAL:  {row['original_text'][:200]}")
    print(f"  AUGMENTED: {str(row['augmented_text'])[:200]}")
    print()

In [ ]:
# ── Restart recovery: reload S4 results from Drive ───────────────────────────
# If the runtime was restarted after S4 completed, uncomment and run this cell
# to reload bt_df before proceeding to the S5 section. Skip if bt_df is already
# in memory.
#
# import pandas as pd
# from pathlib import Path
# RESULTS_DIR = Path("/content/drive/MyDrive/colab/satp-results/augmentation-diagnostics")
# bt_df = pd.read_csv(RESULTS_DIR / "s4_backtranslation_results.csv")
# print(f"Reloaded bt_df: {len(bt_df)} rows")

## 5. S5 — T5 Paraphrase

Uses `T5ParaphraseAugmentation` from the classification paper with `google/flan-t5-large` — the same model used in the classification paper experiments.

T5 paraphrase is expected to have a lower count-preservation rate than back-translation: T5 may convert numerals to words ("three"), drop numbers in long compound sentences, or reorder clauses in ways that lose the count. The failure-type breakdown below distinguishes these cases.

### 5.1 Load T5 paraphrase model

In [ ]:
from imbalance_handling_strategies import T5ParaphraseAugmentation

print(f"Loading T5 paraphrase model: {T5_PARAPHRASE_MODEL}")
print("This may take a minute on first run (model download + load)...")
t5_aug = T5ParaphraseAugmentation(model_name=T5_PARAPHRASE_MODEL)
print("Model loaded.")

### 5.2 Run T5 paraphrase diagnostic

In [ ]:
# ── Run T5 paraphrase diagnostic ─────────────────────────────────────────────
t5_records = []
print(f"Generating {N_PARAPHRASES} paraphrases per example for {len(sample_df)} examples...")

for i, (_, row) in enumerate(sample_df.iterrows()):
    if i % 10 == 0:
        print(f"  {i}/{len(sample_df)}...")
    try:
        paraphrases = t5_aug.paraphrase(
            row['incident_summary'],
            num_return_sequences=N_PARAPHRASES,
            seed=RANDOM_SEED + i
        )
    except Exception as e:
        paraphrases = []
        print(f"  Error on row {i}: {e}")

    for j, para in enumerate(paraphrases):
        t5_records.append({
            'incident_number':  row['incident_number'],
            'bin':              row['bin'],
            'total_fatalities': row['total_fatalities'],
            'original_text':    row['incident_summary'],
            'paraphrase_index': j,
            'augmented_text':   para,
        })

t5_df = pd.DataFrame(t5_records)
t5_df = run_preservation_check(t5_df)
print(f"\nTotal T5 paraphrase results: {len(t5_df)}")

### 5.3 Pass rates

In [ ]:
# ── S5 pass rates ────────────────────────────────────────────────────────────
print("=== S5 Overall pass rate ===")
overall = t5_df['preserved'].agg(['sum', 'count'])
print(f"  {overall['sum']}/{overall['count']} ({100*overall['sum']/overall['count']:.1f}%)")

print("\n=== S5 Pass rate by bin ===")
print(summarise_pass_rates(t5_df, ['bin']).to_string(index=False))

### 5.4 Failure mode classification

In [ ]:
# ── S5 failure mode classification ───────────────────────────────────────────
def classify_failure(row):
    aug = row['augmented_text']
    label = row['total_fatalities']
    if aug is None or not isinstance(aug, str) or len(aug.strip()) == 0:
        return 'generation_failed'
    num_words = {
        1:'one', 2:'two', 3:'three', 4:'four', 5:'five',
        6:'six', 7:'seven', 8:'eight', 9:'nine', 10:'ten'
    }
    word = num_words.get(label, '')
    if word and re.search(r'\b' + word + r'\b', aug, re.IGNORECASE):
        return 'numeral_to_word'  # number present but as a word
    if re.search(r'\d+', aug):
        return 'wrong_number'     # some number present but not the right one
    return 'number_dropped'

t5_failures = t5_df[~t5_df['preserved']].copy()
t5_failures['failure_type'] = t5_failures.apply(classify_failure, axis=1)

print(f"Total failures: {len(t5_failures)} of {len(t5_df)} ({100*len(t5_failures)/len(t5_df):.1f}%)")
print("\nFailure type breakdown:")
print(t5_failures['failure_type'].value_counts())

### 5.5 Failure inspection

In [ ]:
# ── S5 failure inspection ────────────────────────────────────────────────────
for _, row in t5_failures.head(INSPECT_N).iterrows():
    print(f"[bin={row['bin']} | label={row['total_fatalities']} | type={row['failure_type']}]")
    print(f"  ORIGINAL:  {row['original_text'][:200]}")
    aug = row['augmented_text']
    print(f"  PARAPHRASE:{str(aug)[:200] if aug else 'GENERATION FAILED'}")
    print()

### 5.6 Passing example inspection

In [ ]:
# ── S5 passing example inspection ────────────────────────────────────────────
t5_passes = t5_df[t5_df['preserved']]
inspect_t5_passes = (
    t5_passes
    .groupby('bin', group_keys=False)
    .apply(lambda g: g.sample(min(3, len(g)), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

for _, row in inspect_t5_passes.iterrows():
    print(f"[bin={row['bin']} | label={row['total_fatalities']}]")
    print(f"  ORIGINAL:  {row['original_text'][:200]}")
    print(f"  PARAPHRASE:{str(row['augmented_text'])[:200]}")
    print()

## 6. Summary

### 6.1 Comparison table

In [ ]:
# ── Side-by-side comparison table ───────────────────────────────────────────
rows = []

for lang in ['hi', 'ur', 'bn']:
    sub = bt_df[bt_df['pivot_language'] == lang]
    if len(sub) == 0:
        continue
    pass_n = sub['preserved'].sum()
    total  = len(sub)
    rows.append({
        'strategy': f'S4 back-translation ({lang})',
        'n_total':   total,
        'n_pass':    pass_n,
        'pass_rate': f"{100*pass_n/total:.1f}%"
    })

pass_n = t5_df['preserved'].sum()
total  = len(t5_df)
rows.append({
    'strategy': f'S5 T5 paraphrase ({T5_PARAPHRASE_MODEL})',
    'n_total':   total,
    'n_pass':    pass_n,
    'pass_rate': f"{100*pass_n/total:.1f}%"
})

summary_df = pd.DataFrame(rows)
print("=== Augmentation count-preservation summary ===")
print(summary_df.to_string(index=False))

print()
print("Interpretation:")
print("  >= 85%: viable — proceed to integration")
print("  70-85%: marginal — consider best-performing languages only")
print("  < 70%:  adds more noise than signal — do not integrate")

### 6.2 Coverage estimate

In [ ]:
# ── Coverage estimate ────────────────────────────────────────────────────────
n_rare_total = len(rare_df)
print(f"Total rare-bin examples in train set: {n_rare_total}")
print("Projected augmented examples passing count-preservation (full rare-bin set):")

for _, row in summary_df.iterrows():
    rate = float(row['pass_rate'].strip('%')) / 100
    if 'T5' in row['strategy']:
        projected = int(n_rare_total * N_PARAPHRASES * rate)
        print(f"  {row['strategy']}: ~{projected} new examples ({N_PARAPHRASES} per source)")
    else:
        projected = int(n_rare_total * rate)
        print(f"  {row['strategy']}: ~{projected} new examples")

### 6.3 Save results to Drive

In [ ]:
# ── Save results to Drive ────────────────────────────────────────────────────
bt_df.to_csv(RESULTS_DIR / "s4_backtranslation_results.csv", index=False)
t5_df.to_csv(RESULTS_DIR / "s5_t5paraphrase_results.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "augmentation_summary.csv", index=False)
print(f"Results saved to {RESULTS_DIR}")